In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

DATASET_DIR = 'C:/omar/faculty/4- fourth year/GP/Ideas/Video Semantic Search/VRD/datasets/stanford VRD dataset'

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

OBJECTS_JSON = os.path.join(DATASET_DIR, "objects.json")
PREDICATES_JSON = os.path.join(DATASET_DIR, "predicates.json")
TRAIN_ANN = os.path.join(DATASET_DIR, "annotations_train.json")
TEST_ANN = os.path.join(DATASET_DIR, "annotations_test.json")
TRAIN_IMGS = os.path.join(DATASET_DIR, "sg_train_images")
TEST_IMGS = os.path.join(DATASET_DIR, "sg_test_images")

print("Objects vocab file :", OBJECTS_JSON)
print("Predicates vocab file:", PREDICATES_JSON)
print("Train annotations :", TRAIN_ANN)
print("Test annotations :", TEST_ANN)
print("Train images dir :", TRAIN_IMGS)
print("Test images dir :", TEST_IMGS)

---
## Load Vocab & Explore Annotations

In [ ]:
import json
from collections import Counter

with open(OBJECTS_JSON) as f:
    OBJECT_CLASSES = json.load(f)
with open(PREDICATES_JSON) as f:
    PREDICATE_CLASSES = json.load(f)

print(f"Object classes : {len(OBJECT_CLASSES)}. {OBJECT_CLASSES[:10]}")
print(f"Predicate classes : {len(PREDICATE_CLASSES)}. {PREDICATE_CLASSES[:10]}")

In [ ]:
with open(TRAIN_ANN) as f: raw_train = json.load(f)
with open(TEST_ANN) as f: raw_test = json.load(f)

print(f"Train images : {len(raw_train):,}")
print(f"Test images : {len(raw_test):,}")
print(f"Train relationships: {sum(len(v) for v in raw_train.values()):,}")
print(f"Test relationships: {sum(len(v) for v in raw_test.values()):,}")

first_key = next(k for k, v in raw_train.items() if v)
sample_rels = raw_train[first_key]
print(f"\nSample filename : {first_key}")
print(f"Relationships in it: {len(sample_rels)}")
print(f"Relationship keys : {list(sample_rels[0].keys())}")
print(f"\nSample relationship: {json.dumps(sample_rels[0], indent=2)}")

r0 = sample_rels[0]
print(
    f"\nResolved: "
    f"{OBJECT_CLASSES[r0['subject']['category']]} "
    f"{PREDICATE_CLASSES[r0['predicate']]} "
    f"{OBJECT_CLASSES[r0['object']['category']]}"
)

---
## Convert Annotations -> Pipeline Format


In [ ]:
from tqdm.auto import tqdm
from PIL import Image


def _yxyx_to_xyxy(bbox):
    y1, y2, x1, x2 = bbox
    return [float(x1), float(y1), float(x2), float(y2)]


def convert_vrd_annotations(
    raw_ann:    dict,
    images_dir: str,
    split_name: str = 'train',
):
    out_ann = {}
    missing = 0
    skipped = 0
    for filename, rels in tqdm(raw_ann.items(), desc=f'Converting {split_name}'):
        img_path = os.path.join(images_dir, filename)
        if not os.path.isfile(img_path):
            missing += 1
            continue

        try:
            with Image.open(img_path) as im:
                W, H = im.size
        except Exception:
            missing += 1
            continue
        if W <= 0 or H <= 0:
            missing += 1
            continue

        obj_key_to_idx = {}
        objects_out    = []

        def _add_object(category_idx, raw_bbox):
            # insert object into objects_out if not already present - return index."
            if category_idx < 0 or category_idx >= len(OBJECT_CLASSES):
                return None
            label = OBJECT_CLASSES[category_idx]
            bbox  = _yxyx_to_xyxy(raw_bbox)
            key   = (label, *bbox)
            if key not in obj_key_to_idx:
                obj_key_to_idx[key] = len(objects_out)
                objects_out.append({
                    'bbox':  bbox,
                    'label': label,
                    'score': 1.0,
                })
            return obj_key_to_idx[key]

        relationships = []
        for r in rels:
            pred_idx = r.get('predicate', -1)
            subj_raw = r.get('subject', {})
            obj_raw  = r.get('object', {})

            if pred_idx < 0 or pred_idx >= len(PREDICATE_CLASSES):
                skipped += 1
                continue
            if 'category' not in subj_raw or 'category' not in obj_raw:
                skipped += 1
                continue

            s_idx = _add_object(subj_raw['category'], subj_raw['bbox'])
            o_idx = _add_object(obj_raw['category'],  obj_raw['bbox'])
            if s_idx is None or o_idx is None:
                skipped += 1
                continue

            relationships.append({
                'subject_idx': s_idx,
                'predicate':   PREDICATE_CLASSES[pred_idx],
                'object_idx':  o_idx,
            })

        img_id = os.path.splitext(filename)[0]
        out_ann[img_id] = {
            'path':          os.path.relpath(img_path, DATASET_DIR),
            'width':         W,
            'height':        H,
            'objects':       objects_out,
            'relationships': relationships,
        }

    print(f"[convert] {split_name}: {len(out_ann):,} images kept, "
          f"{missing} missing/unreadable files, {skipped} malformed relationships skipped")
    return out_ann


print('Converter defined.')

In [ ]:
train_ann_conv = convert_vrd_annotations(raw_train, TRAIN_IMGS, split_name='train')
test_ann_conv  = convert_vrd_annotations(raw_test,  TEST_IMGS,  split_name='test')

CONV_TRAIN = os.path.join('./', 'conv_train.json')
CONV_TEST  = os.path.join('./', 'conv_test.json')

with open(CONV_TRAIN, 'w') as f: json.dump(train_ann_conv, f)
with open(CONV_TEST,  'w') as f: json.dump(test_ann_conv,  f)

print(f"Saved -> {CONV_TRAIN}")
print(f"Saved -> {CONV_TEST}")

In [ ]:
from data.vrd_dataset import VRDDataset

train_ds = VRDDataset(
    annotation_path=CONV_TRAIN,
    object_classes=OBJECT_CLASSES,
    predicate_classes=PREDICATE_CLASSES,
)
test_ds = VRDDataset(
    annotation_path=CONV_TEST,
    object_classes=OBJECT_CLASSES,
    predicate_classes=PREDICATE_CLASSES,
)

print(f"VRDDataset train: {len(train_ds):,} images")
print(f"VRDDataset test : {len(test_ds):,} images")

dist = train_ds.predicate_distribution()
print(f"\nTop-10 predicates:")
for pred, cnt in list(dist.items())[:10]:
    print(f"  {pred:<30} {cnt:>6}")

---
## 7  Train on the Real Dataset

| Classifier | Speed  | Notes                          |
|------------|--------|---------------------------------|
| `rf`       | fast   | Good baseline; parallelises     |
| `lr`       | fast   | Fastest; linear boundaries      |
| `svm`      | slow   | Best generalisation             |


In [ ]:
import cv2, psutil

available_gb = psutil.virtual_memory().available / 1e9
print(f"Available RAM: {available_gb:.1f} GB")
if available_gb < 6.0:
    print("WARNING: low RAM - consider setting MAX_IMAGES=500 below.")

MAX_IMAGES = None   # None = all; set 500 for a quick run

def load_images_for_dataset(dataset, images_base_dir, max_images=None):
    images = {}
    for i, ann in enumerate(tqdm(dataset, desc='Loading images')):
        if max_images and i >= max_images:
            break
        if ann.image_path:
            p = os.path.join(images_base_dir, ann.image_path)
            if os.path.isfile(p):
                img = cv2.imread(p)
                if img is not None:
                    images[ann.image_id] = img
    return images

train_images = load_images_for_dataset(train_ds, DATASET_DIR, MAX_IMAGES)
test_images  = load_images_for_dataset(test_ds,  DATASET_DIR, MAX_IMAGES)

print(f"Loaded {len(train_images):,} train images, {len(test_images):,} test images")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
from models.vrd_model import VRDModel

CLASSIFIER = 'rf'
GLOVE_PATH = 'wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt'
SAVE_DIR   = os.path.join('./', 'checkpoints')

model = VRDModel(
    classifier_name=CLASSIFIER,
    predicate_classes=PREDICATE_CLASSES,
    glove_path=GLOVE_PATH,
    min_score=0.3,
)

print(f"Training {CLASSIFIER.upper()} on {len(train_images):,} images")
t0 = time.time()
model.fit(train_ds, train_images, verbose=True)
print(f"Training complete in {time.time()-t0:.1f}s")

os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SAVE_DIR, f'vrd_{CLASSIFIER}_real.pkl')
model.save(CKPT_PATH)
print(f"Model saved -> {CKPT_PATH}")

---
## 8  Evaluate on the Test Set

In [ ]:
print(f"Evaluating on {len(test_images):,} test images")
t0 = time.time()
preds_all, gts_all = [], []

for ann in tqdm(test_ds, desc='Inference'):
    img = test_images.get(ann.image_id)
    if img is None:
        continue
    preds_all.append(model.predict_annotation(ann, img))
    gts_all.append([
        {
            'subj':     r.subject.label,
            'pred':     r.predicate,
            'obj':      r.object_.label,
            'subj_box': [r.subject.bbox.x1, r.subject.bbox.y1,
                         r.subject.bbox.x2, r.subject.bbox.y2],
            'obj_box':  [r.object_.bbox.x1, r.object_.bbox.y1,
                         r.object_.bbox.x2, r.object_.bbox.y2],
        }
        for r in ann.relationships
    ])

print(f"Inference done in {time.time()-t0:.1f}s")

In [ ]:
from utils.metrics import evaluation_report, print_report

report = evaluation_report(preds_all, gts_all, model.seen_triplets)
print_report(report)

results_dir = os.path.join('./', 'results')
os.makedirs(results_dir, exist_ok=True)
report_path = os.path.join(results_dir, f'eval_{CLASSIFIER}_real.json')
slim = {k: v for k, v in report.items() if not k.startswith('per_pred')}
with open(report_path, 'w') as f:
    json.dump(slim, f, indent=2)
print(f"Report saved -> {report_path}")

---
## 9  Inference on a Single Image

In [ ]:
# model = VRDModel.load(CKPT_PATH)

sample_ann = next(
    ann for ann in test_ds
    if ann.image_id in test_images and len(ann.relationships) >= 2
)
sample_img = test_images[sample_ann.image_id]

print(f"Image : {sample_ann.image_id}  shape={sample_img.shape}")
print(f"GT relationships ({len(sample_ann.relationships)}):")
for r in sample_ann.relationships[:10]:
    print(f"  ({r.subject.label}, {r.predicate}, {r.object_.label})")

boxes  = [[o.bbox.x1, o.bbox.y1, o.bbox.x2, o.bbox.y2] for o in sample_ann.objects]
labels = [o.label for o in sample_ann.objects]
scores = [o.score for o in sample_ann.objects]

triplets = model.predict(sample_img, boxes, labels, scores, top_k=10)

print(f"\nTop-{min(10, len(triplets))} predictions:")
for i, t in enumerate(triplets[:10]):
    print(f"  {i+1:>2}. ({t.subject.label}, {t.predicate}, {t.object_.label})  "
          f"score={t.score:.4f}")

---
## 10  Visualisation

In [ ]:
from utils.visualization import draw_objects, draw_triplets
from matplotlib import pyplot as plt
import random

def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

vis_gt   = draw_objects(sample_img, sample_ann.objects)
vis_pred = draw_triplets(draw_objects(sample_img, sample_ann.objects), triplets, max_draw=5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(bgr_to_rgb(vis_gt));   axes[0].set_title('Detected Objects'); axes[0].axis('off')
axes[1].imshow(bgr_to_rgb(vis_pred)); axes[1].set_title('Top-5 Predicted Relationships'); axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
gallery_anns = [
    ann for ann in test_ds
    if ann.image_id in test_images and len(ann.relationships) >= 1
]
random.shuffle(gallery_anns)
gallery_anns = gallery_anns[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, ann in zip(axes, gallery_anns):
    img    = test_images[ann.image_id]
    boxes  = [[o.bbox.x1, o.bbox.y1, o.bbox.x2, o.bbox.y2] for o in ann.objects]
    labels = [o.label for o in ann.objects]
    scores = [o.score for o in ann.objects]
    trips  = model.predict(img, boxes, labels, scores, top_k=3)
    vis    = draw_triplets(draw_objects(img, ann.objects), trips, max_draw=3)
    top    = f'({trips[0].subject.label}, {trips[0].predicate}, {trips[0].object_.label})' \
             if trips else '(no prediction)'
    ax.imshow(bgr_to_rgb(vis))
    ax.set_title(top, fontsize=9)
    ax.axis('off')

for ax in axes[len(gallery_anns):]:
    ax.set_visible(False)

plt.suptitle('Gallery: Top Predicted Relationship per Image', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from utils.visualization import save_result_grid

grid_images, grid_titles = [], []
for ann in gallery_anns:
    img    = test_images[ann.image_id]
    boxes  = [[o.bbox.x1, o.bbox.y1, o.bbox.x2, o.bbox.y2] for o in ann.objects]
    labels = [o.label for o in ann.objects]
    scores = [o.score for o in ann.objects]
    trips  = model.predict(img, boxes, labels, scores, top_k=3)
    vis    = draw_triplets(draw_objects(img, ann.objects), trips, max_draw=3)
    title  = f'{trips[0].subject.label} -> {trips[0].predicate} -> {trips[0].object_.label}' \
             if trips else ann.image_id
    grid_images.append(vis)
    grid_titles.append(title)

grid_path = os.path.join(results_dir, f'gallery_{CLASSIFIER}_real.jpg')
save_result_grid(grid_images, grid_titles, grid_path, cols=3)
print(f'Gallery saved -> {grid_path}')